# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** SaaS Product Analytics, Cohorts & Customer Lifetime Value (LTV)

---
*Nota Institucional: Este notebook operacionaliza métricas fundacionales de economía unitaria para modelos de negocio basados en hardware interconectado y servicios de suscripción (IoT / SaaS). Utilizando simulaciones de extracciones SQL vía Pandas, el análisis explora el comportamiento transaccional del usuario, derivando tasas de conversión, deserciones por cohorte (Churn Rate), ARPU, y calculando el Lifetime Value esperado segmentando por línea de dispositivo.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette("crest")

### 1. Ingestión del Data Warehouse Simulado (Hardware & Subscriptions)
Cargamos la base transaccional de ventas de dispositivos (Cámaras de Seguridad vs Timbres Inteligentes) y los registros contractuales de almacenamiento en la nube (Suscripciones Premium).

In [ ]:
# Tabla de Dimensiones: Usuarios e Inventario Base Instalada
df_users = pd.read_csv('../data/hardware_users.csv')
df_users['PurchaseDate'] = pd.to_datetime(df_users['PurchaseDate'])

# Tabla de Hechos: Registros de Suscripción
df_subs = pd.read_csv('../data/subscriptions.csv')
df_subs['SubscriptionStart'] = pd.to_datetime(df_subs['SubscriptionStart'])
df_subs['SubscriptionEnd'] = pd.to_datetime(df_subs['SubscriptionEnd'])

# Union Híbrida: Enriquecer base instalada con estatus de suscripción
# Equivalente SQL: SELECT * FROM Users LEFT JOIN Subs ON Users.UserID = Subs.UserID
df_analytics = pd.merge(df_users, df_subs, on='UserID', how='left')

print(f"[*] Base Instalada Total: {len(df_users)} Hardware Activo.")
print(f"[*] Base de Suscriptores: {len(df_subs)} Contratos procesados.")
df_analytics.head()

### 2. Conversiones y Hardware-to-Subscription Attach Rate
El *Attach Rate* mide la eficacia del embudo *in-app* para convertir a un comprador físico reciente en un usuario de valor recurrente.

In [ ]:
total_hardware = len(df_users)
total_subscriptions = len(df_subs)
global_attach_rate = (total_subscriptions / total_hardware) * 100

print(f"[KPI] Global Hardware-to-Subscription Attach Rate: {global_attach_rate:.2f}%")

conv_by_device = df_analytics.groupby('DeviceType').agg(
    Total_Hardware=('UserID', 'count'),
    Suscriptores=('SubscriptionStart', 'count')
)
conv_by_device['Attach_Rate_Pct'] = (conv_by_device['Suscriptores'] / conv_by_device['Total_Hardware']) * 100
conv_by_device


### 3. Customer Churn Rate Anualizado por Segmento
Definimos probabilísticamente la tasa de retención. Operacionalmente: Suscriptores Perdidos / Suscriptores Totales Únicos dentro del periodo evaluado.

In [ ]:
# Sólo evaluamos usuarios que realmente contrataron
df_subscribers = df_analytics[df_analytics['SubscriptionStart'].notna()].copy()

churn_analysis = df_subscribers.groupby('DeviceType').agg(
    Inicios=('UserID', 'count'),
    Cancelaciones=('IsActive', lambda x: (x == False).sum())
)

churn_analysis['Churn_Rate_Pct'] = (churn_analysis['Cancelaciones'] / churn_analysis['Inicios']) * 100

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=churn_analysis.reset_index(), x='DeviceType', y='Churn_Rate_Pct', palette=['#ff5252', '#4db6ac'], ax=ax)
ax.set_title("Customer Churn Rate por Línea de Hardware", fontsize=14, pad=10)
ax.set_ylabel("Índice de Cancelación (% Churn)")
ax.set_xlabel("")
for i, v in enumerate(churn_analysis['Churn_Rate_Pct']):
    ax.text(i, v - 3, f"{v:.1f}%", ha='center', color='white', fontweight='bold', fontsize=12)
plt.show()


### 4. Proyección de LTV (Lifetime Value) y ARPU Segmentado
Se asume un Margen de Contribución Fijo del Cloud Storage (ej: 80%). 
La fórmula rectora: $ LTV = \frac{ARPU \times Margen}{Churn \; Rate} $

In [ ]:
GROSS_MARGIN = 0.80 # 80% margen bruto en software/cloud

# ARPU (Average Revenue Per User mensual) asumiendo cobros lineales
arpu_data = df_subscribers.groupby('DeviceType').agg(ARPU=('MonthlyFee', 'mean'))

# Consolidación Unitaria Económica
economics = pd.concat([arpu_data, churn_analysis[['Churn_Rate_Pct']]], axis=1)
economics['Churn_Decimal'] = economics['Churn_Rate_Pct'] / 100

# Proyección LTV
economics['LTV_Projected_USD'] = (economics['ARPU'] * GROSS_MARGIN) / economics['Churn_Decimal']

economics[['ARPU', 'Churn_Rate_Pct', 'LTV_Projected_USD']].round(2)

### 5. Análisis de Cohortes (Heatmap Estructural)
El análisis de cohortes nos permite auditar si los cambios de producto, marketing o flujos de *onboarding* lanzados en meses específicos impactaron la absorción del Attach Rate a lo largo del tiempo.

In [ ]:
cohort_data = df_analytics.groupby(['CohortMonth', 'DeviceType']).agg(
    Total_Hardware=('UserID', 'count'),
    Suscriptores=('SubscriptionStart', 'count')
).reset_index()

cohort_data['Attach_Rate'] = (cohort_data['Suscriptores'] / cohort_data['Total_Hardware'])

cohort_pivot = cohort_data.pivot(index='CohortMonth', columns='DeviceType', values='Attach_Rate')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cohort_pivot, annot=True, fmt=".1%", cmap="YlGnBu_r", linewidths=.5, cbar_kws={'label': 'Tasa de Conversión a SaaS'})
ax.set_title("Evolución Cohortal del Attach Rate de Suscripción (2022)", fontsize=15, pad=15)
ax.set_ylabel("Cohorte de Adquisición (Mes de Compra)")
ax.set_xlabel("Hardware Core")
plt.show()

### Síntesis para Allocation de Presupuesto (CAC)
Los datos muestran contundentemente que, aunque las *Cámaras de Seguridad* generan cuotas mensuales moderadamente más altas, su **baja propensión a cancelar (Churn Rate)** las dota de un Lifetime Value ($LTV$) casi tres veces superior al de los Timbres Inteligentes. 

Para el equipo de Marketing / Paid Media, esto significa empíricamente que **podemos tolerar un Costo de Adquisición de Clientes (CAC) mucho más agresivo** en las campañas de cámaras que en la línea de timbres, manteniendo un ratio LTV:CAC saludable (> 3:1).